# Test the A/B inference server — using the adapter you already trained

No retraining here. This notebook clones the repo for the `coderefine` code,
then loads the **adapter weights you already produced** (from
`coderefine_results.zip`, the file the training notebook downloaded for you)
into a fresh session, and stands up the real `coderefine serve` A/B API on this
GPU so you can hit it with actual requests.

**Before running: Runtime → Change runtime type → T4 GPU.** Then run the cells
top to bottom — cell 4 will pause and ask you to upload `coderefine_results.zip`
from your computer.

## 1 — GPU check

In [1]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


## 2 — Clone the repo (for the `coderefine` package + configs)

In [2]:
%cd /content
!rm -rf loRA-code-refinement
!git clone -q https://github.com/hamnaraeel/loRA-code-refinement.git
%cd loRA-code-refinement

/content
/content/loRA-code-refinement


## 3 — Install

Same pinned combo used for training — `bitsandbytes` needs this exact
`transformers`/`peft`/`trl`/`accelerate` set to load a 4-bit adapter correctly.

In [3]:
!pip install -q "transformers==4.46.3" "peft==0.13.2" "trl==0.11.4" "accelerate==1.2.1" "tokenizers<0.21" \
    "bitsandbytes>=0.43.1" "datasets>=2.19" fastapi uvicorn httpx pyyaml pydantic typer rich
!pip install -q -e . --no-deps

import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 115.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 83.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 22.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingfa

## 4 — Upload your trained adapter

This cell pauses and shows a file picker — choose `coderefine_results.zip`,
the file the training notebook downloaded to your computer. It unzips it,
finds the adapter inside (wherever it landed in the zip), and copies it into
`artifacts/runs/qlora-mistral7b-r16/adapter/` — the path the server expects.

In [ ]:
from google.colab import files
import pathlib, shutil, zipfile

print("Upload coderefine_results.zip (downloaded by the training notebook).")
uploaded = files.upload()
zip_name = next(iter(uploaded))

extract_dir = pathlib.Path("/content/coderefine_results_upload")
if extract_dir.exists():
    shutil.rmtree(extract_dir)
extract_dir.mkdir(parents=True)

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(extract_dir)

hits = list(extract_dir.rglob("adapter_config.json"))
if not hits:
    raise RuntimeError(
        f"No adapter_config.json found inside {zip_name!r}. Make sure you "
        "uploaded the coderefine_results.zip produced by the training notebook."
    )

found = hits[0].parent
print("Found adapter at:", found)

target = pathlib.Path("artifacts/runs/qlora-mistral7b-r16/adapter")
target.mkdir(parents=True, exist_ok=True)
shutil.copytree(found, target, dirs_exist_ok=True)

!ls -la artifacts/runs/qlora-mistral7b-r16/adapter/

## 5 — Start the A/B server

Runs as a true background process so this cell returns immediately instead of
blocking the notebook — the next cell polls `/healthz` until it's ready.

In [ ]:
import subprocess, time, requests

BASE = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER = "artifacts/runs/qlora-mistral7b-r16/adapter"

proc = subprocess.Popen(
    ["coderefine", "serve", "--base-model", BASE, "--adapter", ADAPTER, "--load-in-4bit", "--port", "8000"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

for _ in range(90):
    if proc.poll() is not None:
        print(proc.stdout.read())
        raise RuntimeError("Server process exited early — see log above.")
    try:
        r = requests.get("http://127.0.0.1:8000/healthz", timeout=2)
        if r.ok:
            print(r.json())
            break
    except Exception:
        time.sleep(2)
else:
    print(proc.stdout.read())
    raise RuntimeError("Server did not become healthy in time — see log above.")

## 6 — Hit the real /ab endpoint

Both models, one prompt, automatic scores — the actual production behavior,
not a benchmark script.

In [ ]:
import requests, json

payload = {
    "lang": "py",
    "old_code": "def load_config(path):\n    try:\n        with open(path) as fh:\n            return json.load(fh)\n    except Exception:\n        return None",
    "comment": "This except block swallows the error and returns None, which hides real failures. Just let it propagate.",
}
r = requests.post("http://127.0.0.1:8000/ab", json=payload, timeout=120)
print(json.dumps(r.json(), indent=2))

## 7 — Try your own example

Edit `old_code`, `comment`, and `lang` below and re-run to test any case you
want.

In [ ]:
payload = {
    "lang": "py",
    "old_code": "REPLACE ME",
    "comment": "REPLACE ME",
}
r = requests.post("http://127.0.0.1:8000/ab", json=payload, timeout=120)
print(json.dumps(r.json(), indent=2))

## 8 — View the HTML demo console in-browser

Colab can proxy a port running inside the session directly into an iframe —
this is the `GET /` console mentioned in the README.

In [ ]:
from google.colab.output import serve_kernel_port_as_iframe
serve_kernel_port_as_iframe(8000, path="/")

## 9 — Shut the server down when done

In [ ]:
proc.terminate()
print("stopped")